# Orbit Wars Beginner Helper Functions

🚀 Greetings space captains! 🚀

I figured I'd put together a notebook of useful helper functions I've utilized in my [Proto](https://www.kaggle.com/code/djenkivanov/orbit-wars-agent-ow-proto-passed-1-000) agent, as well as further refined in my private notebooks.

Some of the helpers are dependant on other helpers. Also some code samples include made-up function/variable names, but the gist of the code should still be obvious.

I did my best to sort the functions in ascending difficulty.

If there are any mistakes, let me know. I'll do my best to fix it ASAP. I'm also planning to update this with more helpers soon.

Any and all advice/feedback is appreciated!

## Imports

Make sure you have math and orbit wars modules imported as the helper functions in this notebook utilize them.

In [ ]:
import math
import kaggle_environments.envs.orbit_wars.orbit_wars as ow

## Calculate fleet speed

Using this helper you can calculate the fleet speed of a fleet by using its ships amount.

In [ ]:
MAX_SPEED = 6.0

def get_fleet_speed(ships):
    return 1.0 + (MAX_SPEED - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5

# ===================================================================

# USAGE
for f in fleets:
    fleet_speed = get_fleet_speed(f.ships)
    # logic that needs fleet_speed...

## Distance between planets

For this function it's just Pythagorean theorem. Bear in mind the order of `a` and `b` is **sensitive**, as the function says, distance from a to b. It's sensitive because we subtract the launching planet's radius from the total distance.

We do `- a.radius` because we launch planets from the edge of our planets, not from the center point.

In [ ]:
def dist_from(a, b):
    return math.sqrt( (a.x - b.x)**2 + (a.y - b.y)**2 ) - a.radius

# ===================================================================

# USAGE
DIST_THRESHOLD_ATACK = 20

for t in targets:
    for m in mine:
        if dist_from(m, t) < DIST_THRESHOLD_ATACK:
            # attack logic...

## Angle from Planet A to Planet B

Just as `dist_from` was order sensitive, so is `angle_from`. The angle is calculated from A to B, so A would be our planet where we'd want to launch the attack, and B would be the target.

In [ ]:
def angle_from(a, b):
    return math.atan2(b.y - a.y, b.x - a.x)

# ===================================================================

# USAGE
angle = angle_from(my_planet, target)
required_ships = req_ships_capture(my_planet, target)
moves.append([my_planet.id, angle, required_ships])

## Quality of Life local obs helper

I do this once at the beginning of each tick/round/step of the game. It's useful in my opinion, as it saves a lot of redundant duplicate code everytime I need my planets, or targets etc...

In [ ]:
def get_local_obs(obs):
    planets = [ow.Planet(*p) for p in obs.get("planets", [])]
    mine = [p for p in planets if p.owner == obs.get("player", [])]
    targets = [p for p in planets if p.owner != obs.get("player", [])]
    player = obs.get("player", -2)
    fleets = [ow.Fleet(*f) for f in obs.get("fleets", [])]

    return {
        "planets": planets,
        "mine": mine,
        "targets": targets,
        "player": player,
        "fleets": fleets
    }

# ===================================================================

# USAGE
def agent(obs):
    lobs = get_local_obs(obs)
    optimal_target = function_for_optimal_target(
        lobs.get("mine", []),
        lobs.get("targets", [])
    )

## Orbiting planets

Both methods are valid, use whichever one you'd prefer. The competition docs state that orbiting planets are defined as `orbital_radius + planet_radius < 50`, in case you were wondering where that came from.

In [ ]:
# METHOD 1: ARRAY WITH ORBITING PLANET IDS (run once at the beginning of the game to fill orbiting_planets)
orbiting_planets = set()

SUN_X = 50
SUN_Y = 50

def fill_orbiting_planets(obs):
    initial = {i[0]: ow.Planet(*i) for i in obs.get("initial_planets", [])}
    
    for p_id, p in initial.items():
        orbital_radius = math.sqrt( (p.x - SUN_X)**2 + (p.y - SUN_Y)**2 )
        
        if (orbital_radius + p.radius) < 50:
            orbiting_planets.add(p_id)

# ===================================================================

# USAGE
planets = [ow.Planet(*p) for p in obs.get("planets", [])]
for p in planets:
    if p.id in orbiting_planets:
        # orbiting planet logic
    else:
        # static planet logic

In [ ]:
# METHOD 2: BOOLEAN
SUN_X = 50
SUN_Y = 50

def is_orbiting(p):
    orbital_radius = math.sqrt( (p.x - SUN_X)**2 + (p.y - SUN_Y)**2 )
    return (orbital_radius + p.radius) < 50

# ===================================================================
    
# USAGE
planets = [ow.Planet(*p) for p in obs.get("planets", [])]
for p in planets:
    if is_orbiting(p):
        # orbiting planet logic
    else:
        # static planet logic

## Nearest friendly planets to target

This function returns you a list of all friendly planets, sorted in ascending distance from given target.  
Could be used to find nearest planet to a target, then launch an attack from that nearest planet towards the target.

In [ ]:
def nearest_planets_to_target(mine, t):
    nearest = []
    
    for m in mine:
        dist = dist_from(m, t)
        nearest.append((m, dist))
    nearest = sorted(nearest, key=lambda k: k[1])
    return nearest

# ===================================================================

# USAGE
nearest = nearest_planets_to_target(
    lobs.get("mine", []), 
    t
)

for n, dist in nearest:
    if n.ships > t.ships:
        # launch attack...

## Simulate planet trajectories

Simulates X,Y positions of given planet for a specific amount of ticks. If no ticks param is passed, it defaults to the `MAX_SIM_TICKS`, which you could define as you wish.

In [ ]:
MAX_SIM_TICKS = 101

def get_planet_trajectories(p, vel, ticks=None):
    planet_trajectories = []
    
    if ticks is None:
        ticks = MAX_SIM_TICKS
        
    angle = math.atan2(p.y - SUN_Y, p.x - SUN_X)
    r = math.sqrt( (p.x - SUN_X)**2 + (p.y - SUN_Y)**2 )
    
    for tick in range(1, ticks):
        angle_t = angle + vel * tick
        x_t = SUN_X + r * math.cos(angle_t)
        y_t = SUN_Y + r * math.sin(angle_t)
        planet_trajectories.append((x_t, y_t))

    return planet_trajectories

# ===================================================================

# USAGE
targets = lobs.get("targets", [])
for t in targets:
    if t.id in orbiting_planets:
        planet_trajectories = get_planet_trajectories(t, obs.angular_velocity)
        for tick, (px, py) in enumerate(planet_trajectories, start=1):
            # logic for calculating attack towards orbiting planet

## Simulate single planet's position

Complementary to the previous helper, this helper simulates a single position of an orbiting (or static) planet. In the case of static, you just get p.x and p.y back.

In [ ]:
def planet_pos_at(p, vel, tick):
    if p.id not in orbiting_planets:
        return p.x, p.y

    angle0 = math.atan2(p.y - 50, p.x - 50)
    r = math.sqrt((p.x - 50) ** 2 + (p.y - 50) ** 2)

    angle_t = angle0 + vel * tick
    return (
        50 + r * math.cos(angle_t),
        50 + r * math.sin(angle_t)
    )

# ===================================================================

# USAGE
planets = lobs.get("planets", [])
for tick in range(1, 10):
    for p in planets:
        px, py = planet_pos_at(p, obs.angular_velocity, tick)
        # logic using simulated planet position...

## Friendly fleet trajectories

Save all friendly fleet trajectories to keep track of where your fleets are headed and how far they are from arriving.  
With every found fleet we decrement `"arrive_tick"`, keeping track of when our fleet will actually arrive to its target.  
After a fleet has collided with a planet, or gone offscreen, *basically stopped existing*, it will no longer be in the `obs` object, and therefore our `fleets` will no longer contain that fleet, so the `not found` branch will trigger, removing the fleet from our memory.

In [ ]:
friendly_fleets = []

def update_friendly_fleet_trajectories(fleets):
    found_fleet_ids = set()
    
    for f_f in friendly_fleets[:]:
        found = False
        
        for f in fleets:
            if f.id in found_fleet_ids:
                continue
            
            if (
                f.from_planet_id == f_f["mine"].id
                and abs(f.angle - f_f["angle"]) < 1e-3 # safety guard in case angles slightly differ cause of float point values
                and f.ships == f_f["ships"]
            ):
                found = True
                found_fleet_ids.add(f.id)
                break

        if found:
            f_f["arrive_tick"] = max(0, f_f["arrive_tick"] - 1)

        if not found:
            friendly_fleets.remove(f_f)

# ===================================================================

# USAGE
def agent(obs):
    moves = []
    lobs = get_local_obs(obs)
    update_friendly_fleet_trajectories(lobs.get("fleets", []))
    targets = lobs.get("targets", [])
    for t in targets:
        # attack logic...
        # when attack is valid, we execute it by appending to 'moves' and add it into fleet trajectories
        if good_attack:
            moves.append([our_planet.id, angle, ships])
            friendly_fleets.append({
                "mine": our_planet,
                "target": t,
                "angle": angle,
                "ships": ships,
                "arrive_tick": arrive_tick,
                "type": "single"
            })        

## Fleet collision with planet

This helper compares the fleet’s movement against the planet’s movement. If, at any moment during the tick, the distance between them becomes smaller than the planet radius, it counts as a collision. This updated helper is more stable and accurate than my previous `collides` helper, which would sometimes miss collisions.

In [ ]:
def collides(A, B, P0, P1, r):
    d0x, d0y = A[0] - P0[0], A[1] - P0[1]
    dvx = (B[0] - A[0]) - (P1[0] - P0[0])
    dvy = (B[1] - A[1]) - (P1[1] - P0[1])

    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - r * r

    if a < 1e-12:
        return c <= 0.0

    disc = b * b - 4.0 * a * c
    if disc < 0.0:
        return False

    sq = math.sqrt(disc)
    t1 = (-b - sq) / (2.0 * a)
    t2 = (-b + sq) / (2.0 * a)
    return t2 >= 0.0 and t1 <= 1.0

# ===================================================================

# USAGE
planets = lobs.get("planets", [])
fleets = lobs.get("fleets", [])

for p in planets:
    collision = False
    
    for f in fleets:
        fleet_speed = get_fleet_speed(f.ships)
        prev_fx = f.x
        prev_fy = f.y
        
        for tick in range(1, MAX_SIM_TICKS):
            next_fx = f.x + math.cos(f.angle) * fleet_speed * tick
            next_fy = f.y + math.sin(f.angle) * fleet_speed * tick
            
            p_old = planet_pos_at(p, obs.angular_velocity, tick-1)
            p_new = planet_pos_at(p, obs.angular_velocity, tick)
            
            if collides(
                (prev_fx, prev_fy),
                (next_fx, next_fy),
                p_old,
                p_new, 
                p.radius
            ):
                # whatever logic you want...
                collision = True
                break

            prev_fx = next_fx
            prev_fy = next_fy

        if collision:
            break

## Sun collision detection

Now that we have the `collides` function, we can use it to detect collisions with the sun.

In [ ]:
SUN_X = 50
SUN_Y = 50
SUN_R = 10

def sun_collision(m, fleet_speed, angle, arrive_tick):
    prev_x = m.x
    prev_y = m.y

    max_ticks = min(arrive_tick + 1, MAX_SIM_TICKS)
    
    for tick in range(1, max_ticks):
        x = m.x + math.cos(angle) * fleet_speed * tick
        y = m.y + math.sin(angle) * fleet_speed * tick

        if collides(prev_x, prev_y, x, y, SUN_X, SUN_Y, SUN_R):
            return True

        prev_x = x
        prev_y = y
            
    return False

# ===================================================================

# USAGE
m, angle, ships, arrive_tick = calculate_sample_attack()
fleet_speed = get_fleet_speed(ships)

if not sun_collision(m, fleet_speed, angle, arrive_tick):
    moves.append([m.id, angle, ships])

## Planets under attack

Things are getting a bit more complex now. This function can be used to find out which planets of a specified group (`watched_planets`) is under attack by enemies. You can pass your own planets as the `watched_planets` parameter, then you'd get a dictionary of all your owned planets which are under attack, as well as extra info about the attackers, such as amount of fleets, arrive ticks, ships amounts. You can also pass all planets as `watched_planets` parameter, to then use that dictionary to your advantage (eg. calculate an opportunity attack to steal a planet right after someone captures it).

In [ ]:
def get_planets_under_attack(watched_planets, fleets, player, vel, planets):
    orb_pl_traj = {}
    under_attack = {}

    p_ids = {p.id for p in watched_planets}
    fleets = [f for f in fleets if f.owner != player]

    for p in planets:
        if p.id in orbiting_planets:
            orb_pl_traj[p.id] = get_planet_trajectories(p, vel)

    for f in fleets:
        fleet_speed = get_fleet_speed(f.ships)

        first_hit = None

        prev_x = f.x
        prev_y = f.y

        hit_planet = None
        arrive_tick = None
        for tick in range(1, MAX_SIM_TICKS):
            next_x = f.x + math.cos(f.angle) * fleet_speed * tick
            next_y = f.y + math.sin(f.angle) * fleet_speed * tick
            
            for p in planets:
                if p.id in orbiting_planets:
                    p_x, p_y = orb_pl_traj[p.id][tick - 1]
                else:
                    p_x, p_y = p.x, p.y

                if collides(prev_x, prev_y, next_x, next_y, p_x, p_y, p.radius):
                    hit_planet = p
                    arrive_tick = tick
                    break

            if hit_planet is not None:
                break

            prev_x = next_x
            prev_y = next_y

        if hit_planet is None or arrive_tick is None:
            continue
        
        if hit_planet.id not in p_ids:
            continue

        if hit_planet.id not in under_attack:
            under_attack[hit_planet.id] = {
                "planet": hit_planet,
                "fleets": []
            }

        under_attack[hit_planet.id]["fleets"].append({
            "fleet": f,
            "arrive_tick": arrive_tick
        })

    return under_attack

# ===================================================================

# USAGE
friendly_under_attack = get_planets_under_attack(
    lobs.get("mine", []),
    lobs.get("fleets", []),
    lobs.get("player", -2),
    obs.angular_velocity,
    lobs.get("planets", [])
)
for p_id, data in friendly_under_attack.items():
    p = data["planet"]
    fleets = data["fleets"]
    
    total_enemy_ships = sum(
        f["fleet"].ships
        for f in friendly_under_attack[p.id]["fleets"]
    )
    
    if total_enemy_ships > p.ships: 
        # reinforcement logic...

## Find valid angle to planet

This helper finds an angle towards your target planet, which avoids the sun, as well as accidental collisions with other planets. If an angle is found, it is returned as well as its arrival tick.

In [ ]:
def find_angle_to_planet(p, t, ships, vel, planets, moving=False):
    fleet_speed = get_fleet_speed(ships)
    angles = []
    
    if moving:    
        for tick in range(1, MAX_SIM_TICKS):
            tx, ty = planet_pos_at(t, vel, tick)
            
            angle = math.atan2(ty - p.y, tx - p.x)
    
            start_x = p.x + math.cos(angle) * (p.radius + 0.1)
            start_y = p.y + math.sin(angle) * (p.radius + 0.1)
    
            dx = tx - start_x
            dy = ty - start_y
    
            dist_to_target = math.sqrt(dx**2 + dy**2)
    
            travel_dist = fleet_speed * tick
            miss_dist = abs(travel_dist - dist_to_target)
    
            if miss_dist > max(t.radius, 1.5):
                continue
    
            if sun_collision(p, fleet_speed, angle, tick):
                continue
    
            angles.append((angle, tick))
        
        for a, arrive_tick in angles:
            start_x = p.x + math.cos(a) * (p.radius + 0.1)
            start_y = p.y + math.sin(a) * (p.radius + 0.1)
        
            prev_x = start_x
            prev_y = start_y
        
            wrong_target = False
        
            for tick in range(1, arrive_tick + 1):
                next_x = start_x + math.cos(a) * fleet_speed * tick
                next_y = start_y + math.sin(a) * fleet_speed * tick
        
                for pl in planets:
                    if pl.id == p.id:
                        continue
        
                    p_old = planet_pos_at(pl, vel, tick - 1)
                    p_new = planet_pos_at(pl, vel, tick)
        
                    if collides(
                        (prev_x, prev_y),
                        (next_x, next_y),
                        p_old,
                        p_new,
                        pl.radius
                    ):
                        if pl.id == t.id:
                            return a, tick
        
                        wrong_target = True
                        break
        
                if wrong_target:
                    break
        
                prev_x = next_x
                prev_y = next_y
                
        return None, None

    else:
        angle = math.atan2(t.y - p.y, t.x - p.x)
    
        start_x = p.x + math.cos(angle) * (p.radius + 0.1)
        start_y = p.y + math.sin(angle) * (p.radius + 0.1)

        prev_x = start_x
        prev_y = start_y
        
        dist = math.sqrt((start_x - t.x)**2 + (start_y - t.y)**2)
        arrive_tick = math.ceil(dist / fleet_speed)
        
        if sun_collision(p, fleet_speed, angle, arrive_tick):
            return None, None

        wrong_target = False
        for tick in range(1, arrive_tick+1):
            next_x = start_x + math.cos(angle) * fleet_speed * tick
            next_y = start_y + math.sin(angle) * fleet_speed * tick

            for pl in planets:
                if pl.id == p.id:
                        continue

                p_old = planet_pos_at(pl, vel, tick - 1)
                p_new = planet_pos_at(pl, vel, tick)
                
                if collides(
                    (prev_x, prev_y),
                    (next_x, next_y),
                    p_old,
                    p_new,
                    pl.radius
                ):
                    if pl.id == t.id:
                        return angle, tick
                        
                    wrong_target = True
                    break

            if wrong_target:
                break

            prev_x = next_x
            prev_y = next_y

    return None, None

# ===================================================================

# USAGE
# SEE CELL RIGHT UNDER THIS ONE

## Defend vulnerable planets by sending reinforcements

These helpers keep track of reinforcement fleets, plan defenses, and execute defenses. Defense plans precisely calculate how many ships are needed by what tick. And we then execute those defense plans using nearby planets, given that they themselves have enough ships to send + defend in case they're under attack themselves.

In [ ]:
MIN_SHIPS_MINE_ATTACK = 5

def update_reinforcement_trajectories():
    for r_t in reinforcement_trajectories[:]:
        r_t["arrive_tick"] -= 1

        if r_t["arrive_tick"] <= 0:
            reinforcement_trajectories.remove(r_t)
            continue


def get_defense_plans(mine, under_attack, comet_planet_ids):
    defense_plans = {}

    for p in mine:
        if p.id not in under_attack or p.id in comet_planet_ids:
            continue

        attacking_fleets = sorted(
            under_attack[p.id]["fleets"],
            key=lambda att: att["arrive_tick"]
        )

        incoming_reinforcements = sorted(
            [r for r in reinforcement_trajectories if r["target"].id == p.id],
            key=lambda r: r["arrive_tick"]
        )

        p_available_ships = p.ships
        previous_tick = 0
        r_idx = 0

        for att in attacking_fleets:
            att_arrive_tick = att["arrive_tick"]

            p_available_ships += (att_arrive_tick - previous_tick) * p.production

            while (
                r_idx < len(incoming_reinforcements)
                and incoming_reinforcements[r_idx]["arrive_tick"] <= att_arrive_tick
            ):
                p_available_ships += incoming_reinforcements[r_idx]["ships"]
                r_idx += 1

            enemy_ships = att["fleet"].ships
            p_available_ships -= enemy_ships
            previous_tick = att_arrive_tick

            if p_available_ships < 0:
                defense_plans[p] = {
                    "type": "defense",
                    "ships_needed": max(
                        MIN_SHIPS_MINE_ATTACK,
                        math.ceil(abs(p_available_ships) + 1)
                    ),
                    "needed_by_tick": att_arrive_tick
                }
                break

    return defense_plans


def execute_defense_plans(defense_plans, mine, under_attack, exhausted_planets_id, vel, planets):
    moves = []
    reserved = {}

    for p, plan in defense_plans.items():
        already_reinforced = any(
            r["target"].id == p.id and r["arrive_tick"] >= 0
            for r in reinforcement_trajectories
        )

        if already_reinforced:
            continue

        ships_needed = plan["ships_needed"]
        needed_by_tick = plan["needed_by_tick"]

        nearest_planets = get_closest_planets_to_target(mine, p)

        for p_np, _ in nearest_planets:
            if p_np.id == p.id or p_np.id in exhausted_planets_id:
                continue

            p_np_available_ships = p_np.ships

            p_np_available_ships -= reserved.get(p_np.id, 0)

            if p_np.id in under_attack:
                enemy_ships = sum(
                    att["fleet"].ships
                    for att in under_attack[p_np.id]["fleets"]
                )
                p_np_available_ships = max(0, p_np_available_ships - enemy_ships)

            sent_reinforcements = max(MIN_SHIPS_MINE_ATTACK, ships_needed)

            if p_np_available_ships < sent_reinforcements:
                continue

            p_moving = p.id in orbiting_planets

            angle, arrive_tick = find_angle_to_planet(
                p_np,
                p,
                sent_reinforcements,
                vel,
                planets,
                moving=p_moving
            )

            if angle is None or arrive_tick is None:
                continue

            if arrive_tick > needed_by_tick:
                continue

            moves.append([p_np.id, angle, sent_reinforcements])
            exhausted_planets_id.add(p_np.id)

            reserved[p_np.id] = reserved.get(p_np.id, 0) + sent_reinforcements
            
            reinforcement_trajectories.append({
                "mine": p_np,
                "target": p,
                "angle": angle,
                "ships": sent_reinforcements,
                "arrive_tick": arrive_tick
            })

            break

    return moves

# ===================================================================

# USAGE
reinforcement_trajectories = []
def agent(obs):
    lobs = refresh_local_obs(obs)
    update_reinforcement_trajectories()
    exhausted_planets_id = set()
    comet_planet_ids = obs.get("comet_planet_ids", [])
    moves = []
    
    under_attack = get_planets_under_attack(
        lobs.get("mine", []), 
        lobs.get("fleets", []),
        lobs.get("player", -2), 
        obs.angular_velocity,
        lobs.get("planets", [])
    )

    defense_plans = get_defense_plans(
        lobs.get("mine", []),
        under_attack,
        comet_planet_ids
    )
    
    defense_moves = execute_defense_plans(
        defense_plans,
        lobs.get("mine", []),
        under_attack,
        exhausted_planets_id,
        obs.angular_velocity,
        lobs.get("planets", [])
    )
    
    moves.extend(defense_moves)